In [ ]:
import torch

# Check if GPU is available
if torch.cuda.is_available():
    # Set the GPU device
    gpu_id = 0  # Replace 0 with the index of the GPU you want to use
    torch.cuda.set_device(gpu_id)
    print(f"Using GPU: {torch.cuda.get_device_name(gpu_id)}")
else:
    print("No GPU available")

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

In [ ]:
# Load the JSON dataset
df_corpus_texts = pd.read_json('dta_corpus_texts.json', lines=True)
df_corpus_tokens = pd.read_json('dta_corpus_tokens.json', lines=True)

# Filter the rows where the year is 1913
filtered_df_texts = df_corpus_texts[df_corpus_texts['year'] == 1913][:5]
filtered_df_tokens = df_corpus_tokens[df_corpus_tokens['year'] == 1913][:5]
# Display the filtered DataFrame
print(filtered_df_texts)

# Optionally, you can save the filtered data to a new file
# filtered_df.to_csv('filtered_data.csv', index=False)

In [ ]:
#drop for one dataframe the year column to have only one year column in merged dataset
filtered_df_tokens_dropped = filtered_df_tokens.drop('year', axis=1)
print(filtered_df_tokens_dropped)

In [ ]:
# Merge DataFrames
combined_df = pd.concat([filtered_df_texts, filtered_df_tokens_dropped], axis=1)

# Display the merged DataFrame
print(combined_df)

In [ ]:
from huggingface_hub import login
login()

In [ ]:
#load model and tokenizer
model_id = "CohereForAI/aya-23-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

In [8]:
# Template 1
template1 = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIAT, PIDAT, PPER,
PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVFIN, VVIMP, VVINF, VVIZU, VVPP, VAFIN, VAIMP, VAINF,
VAPP, VMFIN, VMINF, VMPP, XY, $, , $(, $. und X.  X steht für 'andere'.
Geben Sie das STTS-Tag für die Token des Satzes zurück: {sentence} Der Satz sollte wie folgt tokenisiert werden: {tokens}. 
Geben Sie die Tags im Format TOKEN \t Tag zurück. 
Jedes Token-Tag-Paar sollte in einer neuen Zeile in der Ausgabedatei stehen, also fügen Sie nach den Tags ein Zeilenumbruchzeichen hinzu. 
Geben Sie nur das Token und das Tag aus (keine Erklärung, keine Übersetzung, kein zusätzlicher Text).
"""

In [9]:
#template 2
template2 = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIAT, PIDAT, PPER,
PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVFIN, VVIMP, VVINF, VVIZU, VVPP, VAFIN, VAIMP, VAINF,
VAPP, VMFIN, VMINF, VMPP, XY, $, , $(, $. und X.  X steht für 'andere'.
Tokenisieren Sie die Wörter nicht weiter, sie sind bereits Tokenisiert.
Geben Sie nur den Tag aus (keine Erklärung, keine Übersetzung, kein zusätzlicher Text). Geben Sie den STTS-Tag für die Token zurück: {tokens}.
"""

In [10]:
#template 3
template3 = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIAT, PIDAT, PPER,
PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVFIN, VVIMP, VVINF, VVIZU, VVPP, VAFIN, VAIMP, VAINF,
VAPP, VMFIN, VMINF, VMPP, XY, $, , $(, $. und X.  X steht für 'andere'.
Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Eingabesatz: {tokens} """

In [40]:
#template 4->Template 3 mit Zusatz; das Output keine zusätzlichen Informationen und Erklärungen enthalten soll:
template4 = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIAT, PIDAT, PPER,
PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVFIN, VVIMP, VVINF, VVIZU, VVPP, VAFIN, VAIMP, VAINF,
VAPP, VMFIN, VMINF, VMPP, XY, $, , $(, $. und X.  X steht für 'andere'.
Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. Ihre Antwort soll keine Erklärungen und keine zusätzlichen Informationen enthalten. 
Eingabesatz: {tokens}"""

In [43]:
#template 5->soll nur Liste von POS tags zurückgeben + keine extra Info:
template5 = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIAT, PIDAT, PPER,
PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVFIN, VVIMP, VVINF, VVIZU, VVPP, VAFIN, VAIMP, VAINF,
VAPP, VMFIN, VMINF, VMPP, XY, $, , $(, $. und X.  X steht für 'andere'.
Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von POS-Tags sein, wobei jedes POS-Tag die POS-Markierung von dem Wort aus dem Eingabetext ist. 
Ihre Antwort sollte nur eine Liste von POS-Tags enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes POS-Tag die entsprechende POS-Markierung für ein Wort enthält. 
Eingabesatz: {tokens}"""

In [46]:
#template6->keine extra Info geschrieben als "gib nur eine Liste zurück und nichts anderes"
template6 = """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIAT, PIDAT, PPER,
PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVFIN, VVIMP, VVINF, VVIZU, VVPP, VAFIN, VAIMP, VAINF,
VAPP, VMFIN, VMINF, VMPP, XY, $, , $(, $. und X.  X steht für 'andere'.
Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes. 
Eingabesatz: {tokens}"""

In [49]:
#template 7->englische version des prompts
template7="""You are a German linguist and expert in Part-of-Speech Tagging.
You use STTS (Stuttgart-Tubingen Tagset). 
STTS tags are ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, 
KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIDAT, PIDAT, PPER,PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVVFIN, VVIMP, VVIZU, VVPP, VVPP, VAFIN, VAIMP, VAPP, VMFIN, VMINF, VMPP, XY, $, ,$( , $. and X.
Please provide the POS tags for each word in the input sentence. 
The input will be a list of words in the sentence.
The output format should be a list of tuples, where each tuple consists of a word from the input text
and its corresponding POS tag label.
Your response should include only a list of tuples, in the order that the words appear in the input sentence,
with each tuple containing the corresponding POS tag label for a word.
input: {tokens}"""



In [52]:
#template 8->tagset mit erklärungen
template8= """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Eingabesatz: {tokens}
"""

In [55]:
#template 9->combi of template 6 and template 8
template9= """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes. 
Eingabesatz: {tokens}
"""

In [58]:
#template10->Template 9 + few shot examples
template10= """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes zurück. 
Hier sind ein paar Beispiele, was Sie machen sollen:
Eingabesatz: ['Balde', 'rings', 'die', 'Sterne', 'bleichen', 'Und', 'ermattet', 'von', 'Beschwerde', 'Wächsern', 'ihre', 'Wangen', 'bleichen', '.', 'Fäulnis', 'wittert', 'aus', 'der', 'Erde', '.']
Ausgabe: [('Balde','NN'),('rings','ADV'),('die','ART'),('Sterne','NN'),('bleichen','VVINF'),('Und','KON'),('ermattet','VVFIN'),('von','APPR'),('Beschwerde','NN'),('Wächsern','KOUS'),('ihre','PPOSAT'),('Wangen','NN'),('bleichen','VVINF'),('.','$.'),('Fäulnis','NN'),('wittert','VVFIN'),('aus','APPR'),('der','ART'),('Erde','NN'),('.','$.')]

Eingabesatz: ['Traurig', 'rauscht', 'das', 'Rohr', 'im', 'Tümpel', 'Und', 'sie', 'friert', 'in', 'sich', 'gekauert', '.', 'Fern', 'ein', 'Hahn', 'kräht', '.', 'Übern', 'Tümpel', 'Hart', 'und', 'grau', 'der', 'Morgen', 'schauert', '.']
Ausgabe: [('Traurig','ADJD'), ('rauscht','VVFIN'), ('das','ART'), ('Rohr','NN'), ('im','APPRART'), ('Tümpel','NN'), ('Und','KON'), ('sie','PPER'), ('friert','VVFIN'), ('in','APPR'), ('sich','PRF'), ('gekauert','VVPP'), ('.','$.'), ('Fern','NN'),('ein','ART'), ('Hahn','NN'), ('kräht','VVFIN'), ('.','$.'), ('Übern','NN'), ('Tümpel','NN'), ('Hart','NE'), ('und','KON'), ('grau','ADJD'), ('der','ART'), ('Morgen','NN'), ('schauert','VVFIN'), ('.','$.')]

Eingabesatz: ['In', 'der', 'Schmiede', 'dröhnt', 'der', 'Hammer', 'Und', 'sie', 'huscht', 'am', 'Tor', 'vorüber', '.', 'Glührot', 'schwingt', 'der', 'Knecht', 'den', 'Hammer', 'Und', 'sie', 'schaut', 'wie', 'tot', 'hinüber', '.']
Ausgabe: [('In','APPR'), ('der','ART'), ('Schmiede','NN'), ('dröhnt','VVFIN'), ('der','ART'), ('Hammer','NN'), ('Und','KON'), ('sie','PPER'), ('huscht','VVFIN'), ('am','APPART'), ('Tor','NN'), ('vorüber','PTKVZ'), ('.','$.'), ('Glührot','NN'), ('schwingt','VVFIN'), ('der','ART'), ('Knecht','NN'), ('den','ART'), ('Hammer','NN'), ('Und','KON'), ('sie','PPER'), ('schaut','VVFIN'), ('wie','KOKOM'), ('tot','ADJD'), ('hinüber','ADV'), ('.','$.')]
Eingabesatz: {tokens}
"""

In [34]:
# Function to generate POS tagging, allowing for different templates
def generate_pos_tagging(sentence, tokenized_sentence, template_choice):
    
    # Select the appropriate template based on template_choice
    if template_choice == 1:
        prompt = template1.format(sentence=sentence, tokens=tokenized_sentence)
    elif template_choice == 2:
        prompt = template2.format(tokens=tokenized_sentence)
    elif template_choice == 3:
        prompt = template3.format(tokens=tokenized_sentence)
    elif template_choice == 4:
        prompt = template4.format(tokens=tokenized_sentence)
    elif template_choice == 5:
        prompt = template5.format(tokens=tokenized_sentence)
    elif template_choice == 6:
        prompt = template6.format(tokens=tokenized_sentence)
    elif template_choice == 7:
        prompt = template7.format(tokens=tokenized_sentence)
    elif template_choice == 8:
        prompt = template8.format(tokens=tokenized_sentence)
    elif template_choice == 9:
        prompt = template9.format(tokens=tokenized_sentence)
    elif template_choice == 10:
        prompt = template10.format(tokens=tokenized_sentence)
    else:
        raise ValueError("Invalid template choice! Choose 0, 1, or 2.")
    
    messages = [{"role": "user", "content": prompt}]
    
    # Tokenize the prompt
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    print("tokenized")
    # Generate POS tags
    gen_tokens = model.generate(
        input_ids, 
        max_new_tokens=500,
        do_sample=True, 
        temperature=0.3,
    )
    print("generated pos tags")
    # Decode the generated tokens
    gen_text = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)
    print("finish prompt")
    
    assistant_start = gen_text.find("<|CHATBOT_TOKEN|>") + len("<|CHATBOT_TOKEN|>")
    assistant_response = gen_text[assistant_start:].strip()
    return assistant_response

In [ ]:
# Apply the function to each row in the dataset using the template1
combined_df['generated_pos_tags_with_template1'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=1), axis=1)

# Apply the function to each row in the dataset using the template2
combined_df['generated_pos_tags_with_template2'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=2), axis=1)

# Apply the function to each row in the dataset using the template3
combined_df['generated_pos_tags_with_template3'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=3), axis=1)

# Apply the function to each row in the dataset using the template4
combined_df['generated_pos_tags_with_template4'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=4), axis=1)

# Apply the function to each row in the dataset using the template5
combined_df['generated_pos_tags_with_template5'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=5), axis=1)

# Apply the function to each row in the dataset using the template6
combined_df['generated_pos_tags_with_template6'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=6), axis=1)

# Apply the function to each row in the dataset using the template7
combined_df['generated_pos_tags_with_template7'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=7), axis=1)

# Apply the function to each row in the dataset using the template8
combined_df['generated_pos_tags_with_template8'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=8), axis=1)

# Apply the function to each row in the dataset using the template9
combined_df['generated_pos_tags_with_template9'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=9), axis=1)

# Apply the function to each row in the dataset using the template10
combined_df['generated_pos_tags_with_template10'] = combined_df.apply(lambda row: generate_pos_tagging(row['text'], row['tokens'], template_choice=10), axis=1)

# Print a few rows to verify
print(combined_df.head())

In [38]:
# have to adjust code for each template
import re
# function to extract POS-Tags with Regex
def extract_pos_tags(text):
    return re.findall(r"'([A-Z$]+(?:[A-Z.,(]+)?)'", text)

# function to calculate Accuracy
def calculate_accuracy(row):
    model_tags = extract_pos_tags(row['generated_pos_tags_with_template2'])
    #model_tags = extract_pos_tags(row['assistant_response'])
    expected_tags = row['labels']
    correct_count = sum(1 for model_tag, expected_tag in zip(model_tags, expected_tags) if model_tag == expected_tag)
    return correct_count / len(expected_tags) * 100

In [ ]:
# apply it to every row in the chosen column
accuracy_list = combined_df.apply(calculate_accuracy, axis=1).tolist()

# show results
print(accuracy_list)

